# 01 — LV Scar Exploration

**Focus:** Core Zone (CZ) surfaces only.  
The negative mask of CZ approximates the Border Zone + channel substrate, so CZ is sufficient as the primary representation.

**Steps**
1. Scan dataset → filter cases that have a CZ surface
2. 80 / 20 split — *Known* set (train) and *Held-out* set (untouched)
3. 3-D overlay of all Known CZ surfaces
4. Project each Known CZ to a 2-D polar (bull's-eye) map
5. Superposition: fraction of patients with scar at each polar location

## 0 · Imports & config

In [ ]:
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pyvista as pv
from tqdm import tqdm

# Repo root is one level up from notebooks/
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
from data_loading import scan, PatientCase

# ── user config ───────────────────────────────────────────────────────────────
HD_ROOT     = r"F:/RM_TEKNON_DEVELOP"   # external hard drive
SEED        = 42
TRAIN_RATIO = 0.80
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# polar-map resolution
N_R     = 64    # radial bins  (apex → base)
N_THETA = 128   # angular bins (circumferential)

pv.set_jupyter_backend("static")   # inline PNG renders
plt.rcParams["figure.dpi"] = 120

## 1 · Scan dataset — keep only cases with a CZ surface

In [ ]:
all_cases = scan(HD_ROOT)

valid_cases = [
    c for c in all_cases
    if c.tissue_surfaces.get("core") is not None
    and c.tissue_surfaces["core"].exists()
]

print(f"Total cases scanned : {len(all_cases)}")
print(f"Cases with CZ surface: {len(valid_cases)}")

# quick per-year breakdown
from collections import Counter
year_counts = Counter(c.year for c in valid_cases)
for year, n in sorted(year_counts.items()):
    print(f"  {year}: {n} cases")

## 2 · 80 / 20 split

The split is saved to `results/split.json` so it is fully reproducible.

In [ ]:
random.seed(SEED)
shuffled = random.sample(valid_cases, len(valid_cases))

n_known  = int(len(shuffled) * TRAIN_RATIO)
known    = shuffled[:n_known]
held_out = shuffled[n_known:]

split = {
    "seed":        SEED,
    "train_ratio": TRAIN_RATIO,
    "n_known":     len(known),
    "n_held_out":  len(held_out),
    "known":    [f"{c.year}/{c.patient_id}" for c in known],
    "held_out": [f"{c.year}/{c.patient_id}" for c in held_out],
}
(RESULTS_DIR / "split.json").write_text(json.dumps(split, indent=2))

print(f"Known    : {len(known):3d} cases  → results/split.json")
print(f"Held-out : {len(held_out):3d} cases  (untouched)")

## 3 · 3-D overlay — Known CZ surfaces

All Core Zone meshes are rendered together with per-patient colour and 60 % opacity.

In [ ]:
cmap_3d   = plt.get_cmap("tab20", len(known))
plotter   = pv.Plotter(off_screen=True, window_size=(900, 700))
plotter.set_background("white")

loaded_3d = []  # keep track of which cases rendered OK
for i, case in enumerate(tqdm(known, desc="Loading CZ meshes")):
    try:
        mesh = pv.read(str(case.tissue_surfaces["core"]))
        rgba = [int(x * 255) for x in cmap_3d(i)[:3]] + [150]   # opacity 150/255 ≈ 60%
        plotter.add_mesh(mesh, color=cmap_3d(i)[:3], opacity=0.6,
                         smooth_shading=True, show_scalar_bar=False)
        loaded_3d.append(case)
    except Exception as exc:
        print(f"  ⚠  {case.year}/{case.patient_id} skipped: {exc}")

plotter.view_isometric()
img = plotter.screenshot(None, return_img=True)   # returns np.ndarray
plotter.close()

fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(img)
ax.axis("off")
ax.set_title(f"3-D overlay — Known set  ({len(loaded_3d)} CZ surfaces)", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "3d_overlay_known.png", bbox_inches="tight")
plt.show()
print(f"Saved → results/3d_overlay_known.png")

## 4 · 2-D polar (bull's-eye) projection

Each CZ is projected onto a longitudinal–circumferential polar grid:
- **Radial axis** — apex (centre, `r = 0`) → base (edge, `r = 1`)  
- **Angular axis** — circumferential angle 0–360°

The LV long axis is estimated per-patient via PCA on the best available reference mesh  
(priority: Myocardium → Left Ventricle → Endo Layer → CZ itself).

In [ ]:
# ── helpers ───────────────────────────────────────────────────────────────────

def _ref_mesh(case: PatientCase) -> pv.PolyData:
    """Load the best available reference mesh for long-axis estimation."""
    for key in ("myocardium", "lv", "endo", "epi"):
        p = case.anatomy.get(key)
        if p is not None and p.exists():
            return pv.read(str(p))
    # fallback: use the CZ mesh itself
    return pv.read(str(case.tissue_surfaces["core"]))


def _long_axis(mesh: pv.PolyData):
    """PCA long axis.  Returns (unit_axis, apex_point) where axis points apex→base."""
    pts    = np.array(mesh.points)
    center = pts.mean(axis=0)
    _, _, Vt = np.linalg.svd(pts - center, full_matrices=False)
    axis   = Vt[0]                       # first principal component
    proj   = (pts - center) @ axis
    apex   = pts[proj.argmin()]          # most extreme point in –axis direction
    # guarantee axis points from apex toward base (positive = base side)
    if np.dot(center - apex, axis) < 0:
        axis = -axis
    return axis, apex


def polar_map(case: PatientCase, n_r: int = N_R, n_theta: int = N_THETA) -> np.ndarray:
    """
    Project Core Zone mesh to a 2-D polar grid.

    Returns
    -------
    np.ndarray, shape (n_r, n_theta), dtype bool
        True where CZ mesh has at least one vertex.
        Row 0 = apex, row n_r-1 = base.
    """
    ref  = _ref_mesh(case)
    cz   = pv.read(str(case.tissue_surfaces["core"]))

    axis, apex = _long_axis(ref)

    # Build orthonormal frame (axis, u, w)
    secondary = np.array([0.0, 0.0, 1.0])
    if abs(np.dot(axis, secondary)) > 0.9:
        secondary = np.array([1.0, 0.0, 0.0])
    u = np.cross(axis, secondary);  u /= np.linalg.norm(u)
    w = np.cross(axis, u)

    pts   = np.array(cz.points)
    v     = pts - apex
    s_raw = v @ axis
    span  = s_raw.max() - s_raw.min()
    s     = np.clip((s_raw - s_raw.min()) / (span + 1e-9), 0.0, 1.0)

    perp  = v - np.outer(s_raw, axis)
    theta = np.arctan2(perp @ w, perp @ u) % (2 * np.pi)

    grid = np.zeros((n_r, n_theta), dtype=bool)
    ri   = np.clip((s * n_r).astype(int),                   0, n_r     - 1)
    ti   = np.clip((theta / (2 * np.pi) * n_theta).astype(int), 0, n_theta - 1)
    grid[ri, ti] = True
    return grid


def _polar_ax(grid: np.ndarray, ax, title: str = "", cmap="Reds", vmax=1.0):
    """Draw one bull's-eye map on a pre-created polar Axes."""
    n_r, n_theta = grid.shape
    theta_edges  = np.linspace(0, 2 * np.pi, n_theta + 1)
    r_edges      = np.linspace(0, 1, n_r + 1)
    T, R         = np.meshgrid(theta_edges, r_edges)
    ax.pcolormesh(T, R, grid.astype(float), cmap=cmap,
                  vmin=0, vmax=vmax, shading="flat")
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_rticks([])
    ax.set_xticks([])
    if title:
        ax.set_title(title, fontsize=7, pad=2)

print("Polar-map helpers defined.")

### 4a · Compute polar maps for the Known set

In [ ]:
polar_maps  = {}   # {"year/patient_id": np.ndarray}
failed      = []

for case in tqdm(known, desc="Projecting to polar map"):
    key = f"{case.year}/{case.patient_id}"
    try:
        polar_maps[key] = polar_map(case)
    except Exception as exc:
        failed.append(key)
        print(f"  ⚠  {key}: {exc}")

print(f"\nSuccessful projections: {len(polar_maps)}  |  Failed: {len(failed)}")

### 4b · Individual polar maps — Known set (grid)

In [ ]:
keys = list(polar_maps.keys())
n    = len(keys)
ncols = 8
nrows = int(np.ceil(n / ncols))

fig = plt.figure(figsize=(ncols * 1.5, nrows * 1.5))
fig.suptitle(f"Known set — CZ polar maps  (n = {n})", fontsize=12, y=1.01)

for idx, key in enumerate(keys):
    ax = fig.add_subplot(nrows, ncols, idx + 1, projection="polar")
    _polar_ax(polar_maps[key], ax, title=key.split("/")[1])   # show patient ID only

# hide unused axes
for idx in range(n, nrows * ncols):
    fig.add_subplot(nrows, ncols, idx + 1).axis("off")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "polar_maps_known.png", bbox_inches="tight")
plt.show()
print("Saved → results/polar_maps_known.png")

## 5 · Superposition — scar density map

Each cell shows the **fraction of Known patients** with CZ scar at that polar location.  
Centre = apex, edge = base.

In [ ]:
stack   = np.stack(list(polar_maps.values()), axis=0)   # (N, n_r, n_theta)
density = stack.mean(axis=0)                            # fraction in [0, 1]

fig, axes = plt.subplots(1, 2, figsize=(11, 5),
                         subplot_kw={"projection": "polar"})
fig.suptitle(f"Scar superposition — Known set  (n = {len(polar_maps)})", fontsize=13)

# ── left: binary (any patient has scar there) ─────────────────────────────────
_polar_ax(density > 0, axes[0], cmap="Reds", title="Any-patient scar footprint")

# ── right: continuous density ─────────────────────────────────────────────────
_polar_ax(density, axes[1], cmap="hot_r", vmax=density.max(),
          title="Scar density (fraction of patients)")

# shared colorbar for the density panel
sm = plt.cm.ScalarMappable(cmap="hot_r",
                            norm=plt.Normalize(vmin=0, vmax=density.max()))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes[1], pad=0.12, fraction=0.046)
cbar.set_label("Fraction of patients", fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "superposition_known.png", bbox_inches="tight")
plt.show()
print("Saved → results/superposition_known.png")

---
## Summary

| | |
|---|---|
| Known set | saved in `results/split.json` |
| 3-D overlay | `results/3d_overlay_known.png` |
| Individual polar maps | `results/polar_maps_known.png` |
| Superposition | `results/superposition_known.png` |

> **Note on circumferential alignment.**  
> The long axis is estimated per-patient via PCA; the circumferential reference (`u` direction) is
> derived from a fixed world vector.  If the ADAS3D exports are not all in the same anatomical
> orientation, the superposition will smear angularly.  A registration step (e.g. aligning the
> RV insertion point) would sharpen the density map — this can be added in a future notebook.